# Developmental LM — four-task linear-readout training

This notebook trains one frozen-GRU linear readout per task × checkpoint × initialization seed. It never connects to Google Drive. Upload `adaptation_all_tasks.tsv` and the ZIP containing the 30 Phase 1 checkpoints; all outputs stay under `/content` until the final ZIP download.

Before running, choose **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1. Fresh clone and installation
import pathlib, shutil, subprocess, sys

REPO_URL = "https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git"
PROJECT = pathlib.Path("/content/developmental_checkpoints_word_recognition")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
print("Installed commit:", subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# 2. Confirm CUDA before uploading large files
import torch
assert torch.cuda.is_available(), "CUDA is unavailable. Switch the Colab runtime to T4 GPU and rerun."
print("CUDA:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

In [ ]:
# 3. Upload the single combined stimulus list: adaptation_all_tasks.tsv
from google.colab import files

UPLOAD_DIR = pathlib.Path("/content/devlm_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one stimulus TSV, then rerun this cell.")
stimulus_name, stimulus_bytes = next(iter(uploaded.items()))
STIMULUS_PATH = UPLOAD_DIR / pathlib.Path(stimulus_name).name
STIMULUS_PATH.write_bytes(stimulus_bytes)
print("Stimuli:", STIMULUS_PATH, f"({STIMULUS_PATH.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# 4. Upload the ZIP containing exactly 30 Phase 1 checkpoints
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one checkpoint ZIP, then rerun this cell.")
zip_name, zip_bytes = next(iter(uploaded.items()))
CHECKPOINT_ZIP = UPLOAD_DIR / pathlib.Path(zip_name).name
CHECKPOINT_ZIP.write_bytes(zip_bytes)
CHECKPOINT_DIR = pathlib.Path("/content/devlm_checkpoints")
if CHECKPOINT_DIR.exists():
    shutil.rmtree(CHECKPOINT_DIR)
CHECKPOINT_DIR.mkdir(parents=True)
with zipfile.ZipFile(CHECKPOINT_ZIP) as archive:
    root = CHECKPOINT_DIR.resolve()
    for member in archive.infolist():
        destination = (CHECKPOINT_DIR / member.filename).resolve()
        if root not in destination.parents and destination != root:
            raise ValueError(f"Unsafe ZIP member: {member.filename}")
    archive.extractall(CHECKPOINT_DIR)
print("Extracted checkpoint ZIP to:", CHECKPOINT_DIR)

In [ ]:
# 5. Validate the exact code version, stimuli, checkpoints, and feature table
import collections, devlm
from devlm.adaptation.train import (
    discover_checkpoints, discover_feature_table, load_construction_manifest,
)

items = load_construction_manifest(STIMULUS_PATH)
checkpoints = discover_checkpoints(CHECKPOINT_DIR)
try:
    FEATURE_TABLE_PATH = discover_feature_table(CHECKPOINT_DIR)
    feature_source = "checkpoint ZIP"
except ValueError:
    FEATURE_TABLE_PATH = PROJECT / "colab" / "ipa_feature_mapping.json"
    feature_source = "repository fallback"
print("devlm loaded from:", pathlib.Path(devlm.__file__).resolve())
print("Stimulus counts:", collections.Counter((x.task_name, x.metadata["partition"]) for x in items))
print(f"Feature table ({feature_source}):", FEATURE_TABLE_PATH)
print("Checkpoint order:")
for checkpoint_id, path, hours in checkpoints:
    print(f"  {checkpoint_id}: {hours:8.3f} h  {path.name}")

In [ ]:
# 6. Training settings. These are shared identically across M01–M30.
OUTPUT_DIR = pathlib.Path("/content/devlm_task_adaptation_outputs")
INITIALIZATION_SEEDS = [1729, 2718, 3141]  # paired robustness repetitions, not CV folds
INPUT_NOISE_SEED = 20260904               # identical noisy inputs across checkpoints/seeds
NOISE_SIGMA = 0.05
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 32
ENCODING_BATCH_SIZE = 128
MAX_EPOCHS = 100
PATIENCE = 10
MIN_DELTA = 1e-4
print("Outputs will remain local at:", OUTPUT_DIR)

In [ ]:
# 7. Start training. Progress and one result line per trained head appear live.
# Metrics are rewritten atomically after every head, so rerunning resumes completed heads.
cmd = [
    sys.executable, "-u", "-m", "devlm.adaptation.cli",
    "--stimuli", str(STIMULUS_PATH),
    "--checkpoints-dir", str(CHECKPOINT_DIR),
    "--feature-table", str(FEATURE_TABLE_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--device", "cuda",
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--batch-size", str(BATCH_SIZE),
    "--encoding-batch-size", str(ENCODING_BATCH_SIZE),
    "--max-epochs", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--min-delta", str(MIN_DELTA),
    "--noise-sigma", str(NOISE_SIGMA),
    "--input-noise-seed", str(INPUT_NOISE_SEED),
    "--initialization-seeds", *map(str, INITIALIZATION_SEEDS),
]
print("Running:", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=PROJECT, check=True)

In [ ]:
# 8. Inspect completion: 4 tasks × 30 checkpoints × 3 seeds = 360 rows
import csv
METRICS_PATH = OUTPUT_DIR / "all_task_checkpoint_metrics.tsv"
with METRICS_PATH.open() as handle:
    metrics = list(csv.DictReader(handle, delimiter="\t"))
expected = 4 * 30 * len(INITIALIZATION_SEEDS)
print(f"Completed metric rows: {len(metrics)}/{expected}")
assert len(metrics) == expected, "Training is incomplete; rerun cell 7 to resume."
for task in ("Sound", "Meaning", "Plausibility", "Grammaticality"):
    values = [float(row["test_accuracy"]) for row in metrics if row["task_name"] == task]
    print(f"{task:16s} mean test accuracy across checkpoints/seeds: {sum(values)/len(values):.3f}")

In [ ]:
# 9. ZIP every metric, manifest, and trained head, then download once
archive_base = pathlib.Path("/content/devlm_task_adaptation_outputs")
archive_path = pathlib.Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR))
print("Archive:", archive_path, f"({archive_path.stat().st_size / 1e6:.1f} MB)")
files.download(str(archive_path))